# Setup

Add libraries and configure with Drive

In [ ]:
from google.colab import drive
import pandas as pd
import xml.etree.ElementTree as ET
import os

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# PLABA Data Extraction and Cleanup

In [ ]:
import zipfile
import pandas as pd

zip_path = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/PLABA.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/plaba_unzipped')

train_df = pd.read_csv('/content/plaba_unzipped/train.csv')
val_df = pd.read_csv('/content/plaba_unzipped/val.csv')
test_df = pd.read_csv('/content/plaba_unzipped/test.csv')

plaba_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

plaba_clean = plaba_df[['input_text', 'target_text']].copy()
plaba_clean.columns = ['Medical_Term', 'Lay_Language_Definiton']

plaba_clean['Metadata'] = plaba_df.apply(
    lambda row: {
        'source': 'PLABA',
        'pmid': str(row['pmid']),
        'question': row['question'],
        'adaptation_version': row['Adaptation_Version'],
        'question_type': row['Question_Type']
    }, axis=1
)

plaba_clean.head()

,Medical_Term,Lay_Language_Definiton,Metadata
0,Exercise-Associated Muscle Cramps (EAMC) are a...,Exercise-Associated Muscle Cramps (EAMC) are a...,"{'source': 'PLABA', 'pmid': '29857264', 'quest..."
1,"Background: Muscle cramp is a painful, involun...",Muscle cramps are unconscious contractions of ...,"{'source': 'PLABA', 'pmid': '33722257', 'quest..."
2,Muscle cramp is a temporary but intense and pa...,"Muscle cramp is a temporary but intense, painf...","{'source': 'PLABA', 'pmid': '31696455', 'quest..."
3,Muscular cramp is a common symptom in healthy ...,"Muscle cramps are common in healthy people, es...","{'source': 'PLABA', 'pmid': '30168894', 'quest..."
4,"Muscle cramps result in continuous, involuntar...",Muscle cramps cause constant and unintended co...,"{'source': 'PLABA', 'pmid': '29763070', 'quest..."


# MedLine Plus Glossary Extraction and Cleanup

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import zipfile
import os

medline_zip_path = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/MedLineGlossary.zip'
extract_folder = '/content/medline_unzipped'

with zipfile.ZipFile(medline_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

xml_file_path = None
for root_dir, dirs, files in os.walk(extract_folder):
    for file in files:
        if file.endswith('.xml'):
            xml_file_path = os.path.join(root_dir, file)
            break
    if xml_file_path:
        break

if xml_file_path:
    tree = ET.parse(xml_file_path)
    root = tree.getroot()

    for elem in root.iter():
        if '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]

    medline_records = []

    for topic in root.findall('.//health-topic'):
        # ONLY process English topics - we do not need spanish for now
        if topic.attrib.get('language') != 'English':
            continue

        term_title = topic.attrib.get('title', 'Unknown Term')
        summary_element = topic.find('.//full-summary')

        if summary_element is not None:
            summary_text = "".join(summary_element.itertext()).strip()

            if summary_text:
                medline_records.append({
                    'Medical_Term': term_title,
                    'Lay_Language_Definition': summary_text,
                    'Metadata': {
                        'source': 'MedlinePlus',
                        'url': topic.attrib.get('url', 'No URL')
                    }
                })

    medline_df = pd.DataFrame(medline_records)
    master_df = pd.concat([plaba_clean, medline_df], ignore_index=True)

    print(f"Added {len(medline_df)} English MedlinePlus records.")
    print(f"Total dictionary size so far: {len(master_df)}")

Added 1015 English MedlinePlus records.
Total dictionary size so far: 1936


# MedQuAd Extraction and Clean

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

# GitHub master folder
medquad_base = '/content/medquad_unzipped/MedQuAD-master'

medquad_records = []
xml_files_found = 0
qa_pairs_extracted = 0

print("Scanning MedQuAD folders for Q&A pairs...")

# go through every subfolder in MedQuAD-master
for root_dir, dirs, files in os.walk(medquad_base):
    for file in files:
        if file.endswith('.xml'):
            xml_files_found += 1
            file_path = os.path.join(root_dir, file)

            try:
                tree = ET.parse(file_path)
                root = tree.getroot()

                # main medical topic of the file
                focus_node = root.find('.//Focus')
                focus = focus_node.text if focus_node is not None else "Unknown"

                # Find every Question-Answer pair in the document
                for qa in root.findall('.//QAPair'):
                    question = qa.find('Question')
                    answer = qa.find('Answer')

                    if question is not None and answer is not None and answer.text:
                        qa_pairs_extracted += 1
                        medquad_records.append({
                            'Medical_Term': question.text.strip(),
                            'Lay_Language_Definition': answer.text.strip(),
                            'Metadata': {
                                'source': 'MedQuAD',
                                'topic_focus': focus
                            }
                        })
            except Exception as e:
                continue # skip any bad files

print(f"Diagnostics: Found {xml_files_found} XML files.")
print(f"Diagnostics: Extracted {qa_pairs_extracted} Q&A pairs.")

# Append new records to master dataframe
if qa_pairs_extracted > 0:
    medquad_df = pd.DataFrame(medquad_records)
    final_master_df = pd.concat([master_df, medquad_df], ignore_index=True)
    print(f"\nSUCCESS! Final dictionary size ready for embeddings: {len(final_master_df)}")
else:
    print("\nFAILURE: No Q&A pairs found.")

final_master_df.tail()

Scanning MedQuAD folders for Q&A pairs...
Diagnostics: Found 11274 XML files.
Diagnostics: Extracted 16407 Q&A pairs.

SUCCESS! Final dictionary size ready for embeddings: 18343


,Medical_Term,Lay_Language_Definiton,Metadata,Lay_Language_Definition
18338,What is (are) succinic semialdehyde dehydrogen...,NaN,"{'source': 'MedQuAD', 'topic_focus': 'succinic...",Succinic semialdehyde dehydrogenase deficiency...
18339,How many people are affected by succinic semia...,NaN,"{'source': 'MedQuAD', 'topic_focus': 'succinic...",Approximately 350 people with succinic semiald...
18340,What are the genetic changes related to succin...,NaN,"{'source': 'MedQuAD', 'topic_focus': 'succinic...",Mutations in the ALDH5A1 gene cause succinic s...
18341,Is succinic semialdehyde dehydrogenase deficie...,NaN,"{'source': 'MedQuAD', 'topic_focus': 'succinic...",This condition is inherited in an autosomal re...
18342,What are the treatments for succinic semialdeh...,NaN,"{'source': 'MedQuAD', 'topic_focus': 'succinic...",These resources address the diagnosis or manag...


# Embeddings.

 Uses batching to not exceed the limits of the free Google AI API rate limit. Results are saved in a JSONL file in the "FaithfulMed Data Simplifier" folder.

In [ ]:
#installls genAI library
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata
import time
import json

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Combine Term and Definition for the embedding model
print("Preparing text for embeddings...")
final_master_df['Text_to_Embed'] = "Term: " + final_master_df['Medical_Term'].astype(str) + " | Definition: " + final_master_df['Lay_Language_Definition'].astype(str)

# Define where the final embeddings will be
output_file = '/content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/RAG_Embeddings_Full.jsonl'
batch_size = 100
requests_per_minute = 14 # Slightly under 15 to meet limit
sleep_time = 60 / requests_per_minute

print(f"Starting batch processing. Total rows: {len(final_master_df)}")

# Process in chunks and write to Drive immediately
with open(output_file, 'a') as f:
    for i in range(0, len(final_master_df), batch_size):
        batch_df = final_master_df.iloc[i:i+batch_size].copy()
        text_list = batch_df['Text_to_Embed'].tolist()

        try:
            # Request embeddings for the whole batch
            response = genai.embed_content(
                model="models/text-embedding-004",
                content=text_list,
                task_type="retrieval_document"
            )

            # Attach vectors back to the chunk
            batch_df['Embedding'] = response['embedding']

            # Save progress line by line to JSONL format
            for _, row in batch_df.iterrows():
                f.write(row.to_json() + '\n')

            print(f"Successfully processed rows {i} to {i + len(batch_df) - 1}")

        except Exception as e:
            print(f"Error on rows {i} to {i + len(batch_df) - 1}: {e}")

        # pause for free API limits
        time.sleep(sleep_time)

print(f"\nEmbedding generation complete! File saved to {output_file}.")
print("It is now ready to hand off to Vaibhavi")

Preparing text for embeddings...
Starting batch processing. Total rows: 18343


Error on rows 0 to 99: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


Error on rows 100 to 199: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


Error on rows 200 to 299: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


Error on rows 300 to 399: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


...

Error on rows 17900 to 17999: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


Error on rows 18000 to 18099: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


Error on rows 18100 to 18199: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


Error on rows 18200 to 18299: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


Error on rows 18300 to 18342: 400 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.

Embedding generation complete! File saved to /content/drive/MyDrive/Google 2F - FaithfulMed/FaithfulMed Data Simplifier/RAG_Embeddings_Full.jsonl.
It is now ready to hand off to Vaibhavi for ChromaDB ingestion.


# Processing

In [ ]:
import